# 4.3 训练稳定性 (Training Stability)

大规模模型训练中，数值不稳定是常见的失败原因。训练稳定性技术确保模型能顺利完成训练。

本节涵盖：
- 梯度裁剪
- Loss Spike处理
- 权重衰减
- Dropout

## 1. 梯度裁剪

**目的**：防止梯度爆炸导致训练崩溃

**基本原理**：当梯度范数超过阈值时按比例缩放梯度，防止参数更新过大导致训练不稳定。

**核心公式**：
- 如果 ||g|| > max_norm，则 g = g × (max_norm / ||g||)
- 保持梯度方向不变，只缩放幅度
- 常用阈值：1.0（大模型训练标准值）

**为什么需要梯度裁剪**：
- 大模型参数量大，梯度可能在不同层间累积放大
- 某些数据批次可能产生异常大的梯度
- 没有梯度裁剪，一次大的梯度更新可能破坏已学到的知识

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

model = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 64))
x = torch.randn(8, 64)
y = torch.randn(8, 64)

loss = nn.MSELoss()(model(x), y)
loss.backward()

grads_before = []
for p in model.parameters():
    if p.grad is not None:
        grads_before.append(p.grad.clone())

total_norm_before = torch.sqrt(sum(g.norm() ** 2 for g in grads_before))

max_norm = 1.0
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

grads_after = []
for p in model.parameters():
    if p.grad is not None:
        grads_after.append(p.grad.clone())

total_norm_after = torch.sqrt(sum(g.norm() ** 2 for g in grads_after))

print('=== Gradient Clipping ===')
print(f'Gradient norm before clipping: {total_norm_before:.4f}')
print(f'Max norm threshold: {max_norm}')
print(f'Gradient norm after clipping: {total_norm_after:.4f}')

if total_norm_before > max_norm:
    scale = max_norm / total_norm_before
    print(f'Clipped! Scale factor: {scale:.4f}')
else:
    print(f'Not clipped (norm was already below threshold)')

print(f'\nKey: Clipping preserves gradient direction, only scales magnitude.')
print(f'This prevents catastrophic parameter updates from gradient explosions.')

## 2. Loss Spike处理

**目的**：应对训练过程中的损失突增

**基本原理**：检测到loss突然增大时，回滚到之前的检查点，跳过导致问题的数据批次，调整学习率后继续训练。

**Loss Spike的常见原因**：
- 数据质量问题（含异常字符、极长序列等）
- 学习率过大导致参数进入不稳定区域
- 梯度爆炸未被裁剪

**处理策略**：
1. **检测**：监控loss变化率，当loss突增超过阈值时触发
2. **回滚**：恢复到最近的稳定检查点
3. **跳过**：跳过导致问题的数据批次
4. **调整**：降低学习率或增大梯度裁剪阈值

In [ ]:
import torch
import torch.nn as nn
import copy

torch.manual_seed(42)

class LossSpikeHandler:
    def __init__(self, patience=3, spike_threshold=3.0, lr_reduction=0.5):
        self.patience = patience
        self.spike_threshold = spike_threshold
        self.lr_reduction = lr_reduction
        self.best_loss = float('inf')
        self.best_state = None
        self.spike_count = 0

    def check_and_save(self, loss, model_state, lr):
        if loss < self.best_loss:
            self.best_loss = loss
            self.best_state = copy.deepcopy(model_state)
            self.spike_count = 0
            return False, lr

        if loss > self.best_loss * self.spike_threshold:
            self.spike_count += 1
            if self.spike_count >= self.patience:
                new_lr = lr * self.lr_reduction
                self.spike_count = 0
                return True, new_lr
        return False, lr

model = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 1))
handler = LossSpikeHandler(patience=2, spike_threshold=2.0)
lr = 1e-3

losses = [1.0, 0.8, 0.6, 0.5, 5.0, 4.5, 0.4, 0.3, 0.2]

print('=== Loss Spike Handler Demo ===')
for i, loss in enumerate(losses):
    spike, new_lr = handler.check_and_save(loss, model.state_dict(), lr)
    status = 'SPIKE!' if loss > handler.best_loss * handler.spike_threshold else 'OK'
    if spike:
        lr = new_lr
        print(f'  Step {i}: loss={loss:.1f} [{status}] -> ROLLBACK + lr reduced to {lr:.6f}')
    else:
        print(f'  Step {i}: loss={loss:.1f} [{status}] (best={handler.best_loss:.1f})')

print(f'\nKey: Detect loss spikes -> rollback to checkpoint -> reduce learning rate.')
print(f'This prevents training collapse from anomalous data or gradient issues.')

## 3. 权重衰减

**目的**：防止过拟合，提升泛化能力

**基本原理**：在损失函数中添加L2正则化项，惩罚过大的权重值，限制模型复杂度。

**核心公式**：
- L_total = L_task + λ × ||W||²
- λ是权重衰减系数，通常为0.01-0.1
- 在AdamW中，权重衰减与梯度更新解耦

**Adam vs AdamW的区别**：
- Adam：L2正则化加在损失函数中，正则化梯度被自适应学习率缩放
- AdamW：权重衰减直接作用于权重，不受自适应学习率影响，效果更好

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

model = nn.Linear(64, 32)
x = torch.randn(16, 64)
y = torch.randn(16, 32)

optimizer_adamw = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
optimizer_adam = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=0.1)

weight_before = model.weight.clone()

loss = nn.MSELoss()(model(x), y)
loss.backward()
optimizer_adamw.step()
weight_after_adamw = model.weight.clone()

weight_decay_effect = (weight_before - weight_after_adamw) * (1 - 0.1)

print('=== Weight Decay (AdamW) ===')
print(f'Weight norm before update: {weight_before.norm():.4f}')
print(f'Weight norm after update:  {weight_after_adamw.norm():.4f}')
print(f'Weight reduction: {weight_before.norm() - weight_after_adamw.norm():.6f}')

print(f'\nAdamW weight decay formula:')
print(f'  W_new = W_old - lr * (gradient + weight_decay * W_old)')
print(f'  The weight_decay term directly shrinks weights toward zero.')
print(f'\nTypical weight_decay values:')
print(f'  Pretraining: 0.1 (LLaMA, GPT)')
print(f'  Fine-tuning: 0.01 (smaller to preserve pretrained knowledge)')
print(f'\nKey: AdamW decouples weight decay from gradient scaling,')
print(f'making regularization more effective than L2 in Adam.')

## 4. Dropout

**目的**：防止过拟合

**基本原理**：训练时随机将部分神经元的输出置零，迫使模型不依赖特定神经元，提升泛化能力。

**核心特点**：
- 训练时：以概率p随机置零神经元输出
- 推理时：使用所有神经元，输出乘以(1-p)保持期望一致
- 大模型预训练中通常不使用Dropout
- 微调阶段可使用Dropout防止过拟合

**为什么大模型预训练不用Dropout**：
- 数据量极大（万亿token），过拟合风险低
- Dropout会降低训练效率
- 模型参数量大，自身就有很强的正则化效果

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

dropout = nn.Dropout(p=0.3)
x = torch.ones(5, 10)

dropout.train()
x_train = dropout(x)
zero_ratio_train = (x_train == 0).float().mean()
nonzero_mean_train = x_train[x_train != 0].mean()

dropout.eval()
x_eval = dropout(x)
zero_ratio_eval = (x_eval == 0).float().mean()

print('=== Dropout ===')
print(f'Dropout rate p = 0.3')
print(f'\nTraining mode:')
print(f'  Zero ratio: {zero_ratio_train:.2%} (should be ~30%)')
print(f'  Non-zero values: {x_train[x_train != 0][:5].tolist()[:3]}...')
print(f'  Non-zero mean: {nonzero_mean_train:.4f} (scaled by 1/(1-p) = {1/0.7:.4f})')
print(f'  Output mean: {x_train.mean():.4f}')

print(f'\nEval mode:')
print(f'  Zero ratio: {zero_ratio_eval:.2%} (should be 0%)')
print(f'  All values: {x_eval[0].tolist()[:3]}...')
print(f'  Output mean: {x_eval.mean():.4f}')

print(f'\nKey: Dropout zeros ~30% of values during training,')
print(f'scales remaining values by 1/(1-p) to maintain expected output.')
print(f'At inference, no dropout is applied (all neurons active).')
print(f'\nNote: Large LLMs typically do NOT use Dropout during pretraining,')
print(f'but may use it during fine-tuning with small datasets.')